In [29]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

SEQ_LEN = 100

In [30]:
def build_primitive_model():
    inputs = layers.Input(shape=(SEQ_LEN, 1))
    
    x = inputs
    for _ in range(11):  # 11 layers -> receptive field = 100
        x = layers.Conv1D(filters=64, kernel_size=10, padding="same", activation="relu")(x)
    
    outputs = layers.Conv1D(filters=1, kernel_size=1, padding="same", activation='sigmoid')(x)
    
    model = models.Model(inputs, outputs)
    return model

In [31]:
def generate_batch(batch_size):
    x = np.random.rand(batch_size, SEQ_LEN, 1)
    y = x[:, ::-1, :].copy()
    return x, y

In [32]:
model_primitive = build_primitive_model()
model_primitive.compile(
    optimizer="adam",
    loss="mse"
)
model_primitive.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 100, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_89 (Conv1D)              │ (None, 100, 64)        │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_90 (Conv1D)              │ (None, 100, 64)        │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_91 (Conv1D)              │ (None, 100, 64)        │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_92 (Conv1D)              │ (None, 100, 64)        │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_93 (Conv1D)              │ (None, 100, 64)        │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_94 (Conv1D)              │ (None, 100, 64)        │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_95 (Conv1D)              │ (None, 100, 64)        │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_96 (Conv1D)              │ (None, 100, 64)        │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_97 (Conv1D)              │ (None, 100, 64)        │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_98 (Conv1D)              │ (None, 100, 64)        │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_99 (Conv1D)              │ (None, 100, 64)        │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_100 (Conv1D)             │ (None, 100, 1)         │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 411,009 (1.57 MB)

 Trainable params: 411,009 (1.57 MB)

 Non-trainable params: 0 (0.00 B)

In [33]:
for _ in range(1000):
    x_batch, y_batch = generate_batch(32)
    model_primitive.train_on_batch(x_batch, y_batch)

In [34]:
x_test, y_test = generate_batch(1000)

loss = model_primitive.evaluate(x_test, y_test)
print("Test MSE:", loss)

32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - loss: 0.0797
Test MSE: 0.07966165244579315


### Why a Convolutional Model Cannot Properly Reverse a Sequence

A convolutional neural network looks at small parts of the sequence at a time, using filters that only see a few neighboring elements. To reverse a sequence, the model needs to know the exact position of every element and move each one to its opposite end. Since convolutional models only focus on local information and do not easily capture the overall order or long-distance relationships, they cannot reliably learn to reverse a sequence, especially when the sequence is long like in this case. The MSE score is stuck at 0.08 because the model has learned to just predict the most frequent number in the sequence.